In [11]:
import pandas as pd
import pyarrow.parquet as pq
import nltk
import re

from pathlib import Path
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kalle\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kalle\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kalle\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
df = pd.read_csv(
    "data/raw/sms_spam_collection/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"],
)

In [13]:
print("-------------------- HEAD --------------------")
print(df.head(5))
print("-------------------- DESCRIBE --------------------")
print(df.describe())
print("-------------------- INFO --------------------")
print(df.info())


-------------------- HEAD --------------------
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
-------------------- DESCRIBE --------------------
       label                 message
count   5572                    5572
unique     2                    5169
top      ham  Sorry, I'll call later
freq    4825                      30
-------------------- INFO --------------------
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 542.9 KB
None


In [14]:
# Processing for Tokenisation
df = df.drop_duplicates()
df["message"] = df["message"].str.lower()
df["message"] = df["message"].apply(lambda x: re.sub(r"[^a-z\s$!]", "", x))
df["message"] = df["message"].apply(word_tokenize)

In [15]:
print("-------------------- HEAD --------------------")
print(df.head(5))
print("-------------------- DESCRIBE --------------------")
print(df.describe())
print("-------------------- INFO --------------------")
print(df.info())


-------------------- HEAD --------------------
  label                                            message
0   ham  [go, until, jurong, point, crazy, available, o...
1   ham                     [ok, lar, joking, wif, u, oni]
2  spam  [free, entry, in, a, wkly, comp, to, win, fa, ...
3   ham  [u, dun, say, so, early, hor, u, c, already, t...
4   ham  [nah, i, dont, think, he, goes, to, usf, he, l...
-------------------- DESCRIBE --------------------
       label                                            message
count   5169                                               5169
unique     2                                               5094
top      ham  [private, !, your, account, statement, for, sh...
freq    4516                                                  7
-------------------- INFO --------------------
<class 'pandas.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    516

In [16]:
#Processing for smarter tokens
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
df["message"] = df["message"].apply(lambda x: [word for word in x if word not in stop_words])
df["message"] = df["message"].apply(lambda x: [stemmer.stem(word) for word in x])

In [17]:
print("-------------------- HEAD --------------------")
print(df.head(5))
print("-------------------- DESCRIBE --------------------")
print(df.describe())
print("-------------------- INFO --------------------")
print(df.info())

-------------------- HEAD --------------------
  label                                            message
0   ham  [go, jurong, point, crazi, avail, bugi, n, gre...
1   ham                       [ok, lar, joke, wif, u, oni]
2  spam  [free, entri, wkli, comp, win, fa, cup, final,...
3   ham      [u, dun, say, earli, hor, u, c, alreadi, say]
4   ham  [nah, dont, think, goe, usf, live, around, tho...
-------------------- DESCRIBE --------------------
       label                                            message
count   5169                                               5169
unique     2                                               5074
top      ham  [privat, !, account, statement, show, unredeem...
freq    4516                                                  7
-------------------- INFO --------------------
<class 'pandas.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    516

In [18]:
#Rejoining tokens for feature extraction
df["message"] = df["message"].apply(lambda x: " ".join(x))

In [19]:
print("-------------------- HEAD --------------------")
print(df.head(5))
print("-------------------- DESCRIBE --------------------")
print(df.describe())
print("-------------------- INFO --------------------")
print(df.info())

-------------------- HEAD --------------------
  label                                            message
0   ham  go jurong point crazi avail bugi n great world...
1   ham                              ok lar joke wif u oni
2  spam  free entri wkli comp win fa cup final tkt st m...
3   ham                u dun say earli hor u c alreadi say
4   ham          nah dont think goe usf live around though
-------------------- DESCRIBE --------------------
       label                                            message
count   5169                                               5169
unique     2                                               5074
top      ham  privat ! account statement show unredeem point...
freq    4516                                                  7
-------------------- INFO --------------------
<class 'pandas.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5169 

In [20]:
Path("data/interim").mkdir(parents=True, exist_ok=True)

df.to_parquet(path="data/interim/02_cleaned.parquet")